# Statistical significance checks

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import logging

from gsm_benchmarker.results_analyser import MultiVariantMultiModelResultsAnalyser
from gsm_benchmarker.results_analyser.prompt_effect_analyser import PromptEffectAnalyser
from gsm_benchmarker.results_analyser.plotting_utils import plot_glmm, plot_acc_change_distribution

logger = logging.getLogger('notebook')

plt.style.use('seaborn-v0_8-muted')
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
METRIC = "correct"

In [ ]:
pp = Path("../../../../data/gsm-symbolic/outputs").resolve()

p_standard_pilot = pp / "mini_20x50x4__14_11/final"

p_standard = pp / "default_full__06_02/final"

p_code = pp / 'code_full__09_02/corrected'
p_formal = pp / 'formalised_full__16_04/final'

In [ ]:
mres_standard_pilot = MultiVariantMultiModelResultsAnalyser(p_standard_pilot)

mres_standard = MultiVariantMultiModelResultsAnalyser(p_standard)

mres_code = MultiVariantMultiModelResultsAnalyser(p_code)
mres_formal = MultiVariantMultiModelResultsAnalyser(p_formal)


In [ ]:
pea_code = PromptEffectAnalyser(mres_standard, mres_code, "Code output prompt")
pea_formal = PromptEffectAnalyser(mres_standard, mres_formal, "Formalised NL prompt")

### Modelling question difficulty

In [ ]:
_ = pea_code._baseline_mres.plot_question_difficulty_per_model()

In [ ]:
_ = pea_code._baseline_mres.plot_question_difficulty_histogram()

## Question 1
Are the accuracy drops reported in the GSM-Symbolic paper actually significant?

Evaluating significance of accuracy change on 'main' variant vs 'GSM8K' variant with GSM-Symbolic prompt.

### 1A: Pilot run
Checking on a subset of the benchmark first. Projecting results to the full dataset with alpha = 20%.

50/100 questions, 20/50 template variations -> 1000/5000 total questions in the 'main variant' (20% of the benchmark) (+ 50 questions in the original GSM8K variant)

In [ ]:
ALPHA = 0.05
ALPHA_THRESHOLD = 1.1 * ALPHA
PROJECTED_ALPHA = 0.2
PROJECTED_ALPHA_THRESHOLD = 1.1 * PROJECTED_ALPHA

In [ ]:
variant_effect_df_pilot = mres_standard_pilot.analyse_variant_effect('main', metric=METRIC)

In [ ]:
_ = plot_glmm(
    variant_effect_df_pilot,
    'accuracy_change',
    title="Accuracy change due to templates (pilot run)",
    projected_alpha=PROJECTED_ALPHA
)


#### Candidate models
Models which, based on the pilot results, are worth checking on the full dataset.

In [ ]:
candidate_models = variant_effect_df_pilot[variant_effect_df_pilot.p_value < PROJECTED_ALPHA_THRESHOLD].index.get_level_values('model').unique().tolist()
candidate_models

### 1B: Full benchmark
Re-running the tests on the full benchmark (100 + 5000 questions) with the identified candidate models.

In [ ]:
variant_effect_df_full = mres_standard.analyse_variant_effect('main', models=candidate_models, metric=METRIC)
variant_effect_df_full

In [ ]:
_ = plot_glmm(
    variant_effect_df_full,
    'accuracy_change',
    title="Accuracy change due to templates (full benchmark)"
)

In [ ]:
significant_models = variant_effect_df_full[variant_effect_df_full.p_value < ALPHA_THRESHOLD].index.get_level_values('model').unique().tolist()
significant_models

## Question 2
Do alternative prompt formats remove the variant dependency?

Evaluating significance of accuracy change on 'main' variant vs 'GSM8K' variant with code prompt and formalised natural language prompt.

In [ ]:
variant_effect_df_code = mres_code.analyse_variant_effect('main', models=significant_models, metric=METRIC)
_ = plot_glmm(
    variant_effect_df_code,
    'accuracy_change',
    title="Accuracy change due to templates - code output prompt"
)

In [ ]:
variant_effect_df_formal = mres_formal.analyse_variant_effect('main', models=significant_models, metric=METRIC)
_ = plot_glmm(
    variant_effect_df_formal,
    'accuracy_change',
    title="Accuracy change due to templates - formalised NL prompt"
)

### Alternative prompts evaluation
Do alternative prompt formats result in significant performance changes?

Evaluating performance on 'main' variant with the alternative prompts vs GSM-Symbolic prompt.

In [ ]:
acc_change_df_code = pea_code.analyse_accuracy_change_significance(variant='main', models=significant_models, metric=METRIC)
acc_change_df_code

In [ ]:
_ = plot_glmm(
    acc_change_df_code,
    'mean_diff',
    bars_value_ylabel='Accuracy change (mean)',
    title="Effect of code prompt on accuracy on 'main' variant",
    bar_colour='lightblue'
)

In [ ]:
acc_change_df_formal = pea_formal.analyse_accuracy_change_significance(variant='main', models=significant_models, metric=METRIC)
_ = plot_glmm(
    acc_change_df_formal,
    'mean_diff',
    bars_value_ylabel='Accuracy change (mean)',
    title="Effect of formalised NL prompt on accuracy on 'main' variant",
    bar_colour='lightblue'
)

In [ ]:
acc_change_raw = pea_code.get_raw_acc_change(variant='main', metric=METRIC)
_, plot_acc_change_distribution(acc_change_raw, models=significant_models)